# EmpathRAG — DeBERTa NLI Safety Guardrail (Karthik)
**MSML641 · NLP Course Project**  
Fine-tune DeBERTa-v3-base on suicide detection NLI pairs  
Target: Recall > 0.80, Precision > 0.65 on crisis class  
Runtime: **A100 GPU** (Runtime → Change runtime type → A100)  
Expected time: ~2 hours

All NLI CSVs are already on Google Drive — no uploads needed.


In [1]:
# STEP 1: Verify A100
import torch
assert torch.cuda.is_available(), "Switch to A100 runtime before proceeding."
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"Torch: {torch.__version__}")
print("✅ GPU ready.")


GPU  : NVIDIA A100-SXM4-40GB
VRAM : 42.4 GB
Torch: 2.10.0+cu128
✅ GPU ready.


In [2]:
# STEP 2: Install packages
# DeBERTa-v3 requires sentencepiece + protobuf for tokenizer
!pip install -q "accelerate>=1.0.0" evaluate sentencepiece protobuf
import evaluate, accelerate
print(f"evaluate  : {evaluate.__version__}")
print(f"accelerate: {accelerate.__version__}")
print("✅ Packages ready.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
evaluate  : 0.4.6
accelerate: 1.13.0
✅ Packages ready.


In [3]:
# STEP 3: Mount Drive and verify paths
from google.colab import drive
import os
drive.mount("/content/drive")


Mounted at /content/drive


In [4]:

TRAIN_CSV = "/content/drive/MyDrive/Mapped_Data/nli_train.csv"
VAL_CSV   = "/content/drive/MyDrive/Mapped_Data/nli_val.csv"
TEST_CSV  = "/content/drive/MyDrive/Mapped_Data/nli_test.csv"
SAVE_DIR  = "/content/drive/MyDrive/Mapped_Data/safety_guardrail"


In [5]:
import pandas as pd

# Hard assertion — these must match your known-good values exactly
EXPECTED = {"/content/drive/MyDrive/Mapped_Data/nli_train.csv": 185659, "/content/drive/MyDrive/Mapped_Data/nli_val.csv": 23207, "/content/drive/MyDrive/Mapped_Data/nli_test.csv": 23208}
for fname, expected_rows in EXPECTED.items():
    # Use pandas to correctly count rows, handling multiline strings
    actual = len(pd.read_csv(fname, usecols=[0]))
    assert actual == expected_rows, f"WRONG FILE: {fname} has {actual} rows, expected {expected_rows}"
    print(f"  {fname}: {actual:,} rows ✅")

  /content/drive/MyDrive/Mapped_Data/nli_train.csv: 185,659 rows ✅
  /content/drive/MyDrive/Mapped_Data/nli_val.csv: 23,207 rows ✅
  /content/drive/MyDrive/Mapped_Data/nli_test.csv: 23,208 rows ✅


In [6]:
import pandas as pd
import os

os.makedirs(SAVE_DIR, exist_ok=True)

for path in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    assert os.path.isfile(path), f"NOT FOUND: {path}"
    # Use pandas to safely count rows
    rows = len(pd.read_csv(path, usecols=[0]))
    size = os.path.getsize(path) / 1e6
    print(f"  {os.path.basename(path):<20} {rows:>8,} rows  {size:.1f} MB  ✅")

print(f"\nCheckpoint will save to: {SAVE_DIR}")

  nli_train.csv         185,659 rows  146.5 MB  ✅
  nli_val.csv            23,207 rows  18.4 MB  ✅
  nli_test.csv           23,208 rows  18.5 MB  ✅

Checkpoint will save to: /content/drive/MyDrive/Mapped_Data/safety_guardrail


In [7]:
import glob
import os
SAVE_DIR = "/content/drive/MyDrive/Mapped_Data/safety_guardrail"
checkpoints = sorted(glob.glob(f"{SAVE_DIR}/checkpoint-*"))
print("Checkpoints found:", checkpoints)
print("SAVE_DIR exists:", os.path.isdir(SAVE_DIR))
# Also check if there were any prior runs
print("All files in SAVE_DIR:")
for f in sorted(os.listdir(SAVE_DIR)) if os.path.isdir(SAVE_DIR) else []:
    print(" ", f)

Checkpoints found: []
SAVE_DIR exists: True
All files in SAVE_DIR:


In [8]:
# STEP 4: Load NLI CSVs
import pandas as pd
from datasets import Dataset

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(f"Train: {len(train_df):,} rows")
print(f"Val  : {len(val_df):,} rows")
print(f"Test : {len(test_df):,} rows")

# Verify label balance
print(f"\nTrain label distribution:")
print(train_df["nli_label"].value_counts().to_string())
print("  0 = entailment (crisis)  |  1 = contradiction (non-crisis)")

# Verify required columns exist
required = ["text", "hypothesis", "nli_label"]
for col in required:
    assert col in train_df.columns, f"Missing column: {col}"
print("\n✅ All required columns present.")


Train: 185,659 rows
Val  : 23,207 rows
Test : 23,208 rows

Train label distribution:
nli_label
1    92830
0    92829
  0 = entailment (crisis)  |  1 = contradiction (non-crisis)

✅ All required columns present.


In [9]:
# STEP 5: Tokenize
from transformers import AutoTokenizer
from datasets import Dataset

MODEL = "microsoft/deberta-v3-base"
print(f"Loading tokenizer: {MODEL}")
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        batch["hypothesis"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )

train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True, desc="Tokenizing train")
val_ds   = Dataset.from_pandas(val_df).map(tokenize, batched=True, desc="Tokenizing val")
test_ds  = Dataset.from_pandas(test_df).map(tokenize, batched=True, desc="Tokenizing test")

train_ds = train_ds.rename_column("nli_label", "labels")
val_ds   = val_ds.rename_column("nli_label", "labels")
test_ds  = test_ds.rename_column("nli_label", "labels")

# Detect what the tokenizer actually produces and set_format accordingly
sample_keys = list(tokenizer("test premise", "test hypothesis", truncation=True, max_length=256, padding="max_length").keys())
print(f"Tokenizer output keys: {sample_keys}")

format_cols = [k for k in sample_keys if k in ["input_ids", "attention_mask", "token_type_ids"]] + ["labels"]
print(f"set_format columns: {format_cols}")

train_ds.set_format(type="torch", columns=format_cols)
val_ds.set_format(type="torch",   columns=format_cols)
test_ds.set_format(type="torch",  columns=format_cols)

print(f"\nTokenization complete.")
print(f"  Train: {len(train_ds):,}  |  Val: {len(val_ds):,}  |  Test: {len(test_ds):,}")
print("✅ set_format correct.")

Loading tokenizer: microsoft/deberta-v3-base


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Tokenizing train:   0%|          | 0/185659 [00:00<?, ? examples/s]

Tokenizing val:   0%|          | 0/23207 [00:00<?, ? examples/s]

Tokenizing test:   0%|          | 0/23208 [00:00<?, ? examples/s]

Tokenizer output keys: ['input_ids', 'token_type_ids', 'attention_mask']
set_format columns: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']

Tokenization complete.
  Train: 185,659  |  Val: 23,207  |  Test: 23,208
✅ set_format correct.


In [10]:
# STEP 6: Load DeBERTa-v3-base
from transformers import AutoModelForSequenceClassification
import torch

# 2 labels: 0 = entailment (crisis), 1 = contradiction (non-crisis)
id2label = {0: "crisis", 1: "non-crisis"}
label2id = {"crisis": 0, "non-crisis": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    torch_dtype=torch.bfloat16,  # load directly as bf16 — matches A100 training dtype
)
model = model.to("cuda")

# Verify — both must say torch.bfloat16
print(f"Model dtype:      {next(model.parameters()).dtype}")
print(f"Classifier dtype: {model.classifier.weight.dtype}")
print(f"Device:           {next(model.parameters()).device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("\n✅ DeBERTa-v3-base loaded as bf16 on CUDA.")

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight        

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Model dtype:      torch.bfloat16
Classifier dtype: torch.bfloat16
Device:           cuda:0
Model parameters: 184,423,682
Trainable params: 184,423,682

✅ DeBERTa-v3-base loaded as bf16 on CUDA.


In [11]:
# STEP 7: Metrics
# Primary metric is RECALL on crisis class (label 0)
# A missed crisis is the worst possible outcome — recall is paramount
import evaluate
import numpy as np

f1_m        = evaluate.load("f1")
recall_m    = evaluate.load("recall")
precision_m = evaluate.load("precision")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    # pos_label=0 because crisis = label 0
    return {
        "recall":    recall_m.compute(predictions=preds,    references=labels, pos_label=0)["recall"],
        "precision": precision_m.compute(predictions=preds, references=labels, pos_label=0)["precision"],
        "f1":        f1_m.compute(predictions=preds,        references=labels, pos_label=0)["f1"],
    }

print("Metrics: recall + precision + F1 on crisis class (label 0).")
print("Primary metric for best-model selection: recall")


Metrics: recall + precision + F1 on crisis class (label 0).
Primary metric for best-model selection: recall


In [12]:
# import torch
# import numpy as np

# # Take 4 samples from val set — 2 crisis (label 0), 2 non-crisis (label 1)
# crisis_indices     = [i for i, x in enumerate(val_ds) if x["labels"].item() == 0][:2]
# noncrisis_indices  = [i for i, x in enumerate(val_ds) if x["labels"].item() == 1][:2]
# sample_indices = crisis_indices + noncrisis_indices

# model.eval()
# with torch.no_grad():
#     for idx in sample_indices:
#         item = val_ds[idx]
#         input_ids      = item["input_ids"].unsqueeze(0).to(model.device)
#         attention_mask = item["attention_mask"].unsqueeze(0).to(model.device)
#         token_type_ids = item["token_type_ids"].unsqueeze(0).to(model.device)
#         true_label     = item["labels"].item()

#         logits = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids).logits
#         probs  = torch.softmax(logits, dim=-1).squeeze().tolist()
#         pred   = int(torch.argmax(logits, dim=-1).item())

#         print(f"true={true_label} | pred={pred} | p(crisis)={probs[0]:.4f} | p(non-crisis)={probs[1]:.4f} | logits={[round(x,3) for x in logits.squeeze().tolist()]}")

In [13]:
# import torch

# idx = crisis_indices[0]
# item = val_ds[idx]

# print("input_ids shape:      ", item["input_ids"].shape)
# print("attention_mask shape: ", item["attention_mask"].shape)
# print("token_type_ids shape: ", item["token_type_ids"].shape)
# print("labels:               ", item["labels"])

# print("\nFirst 20 input_ids:      ", item["input_ids"][:20].tolist())
# print("Last 20 input_ids:       ", item["input_ids"][-20:].tolist())
# print("attention_mask sum:      ", item["attention_mask"].sum().item(), "(non-padding tokens)")
# print("token_type_ids unique:   ", item["token_type_ids"].unique().tolist())

# # Check a second sample is actually different
# idx2 = noncrisis_indices[0]
# item2 = val_ds[idx2]
# print("\nSample 2 first 20 input_ids:", item2["input_ids"][:20].tolist())
# print("Are input_ids identical between samples?", item["input_ids"].equal(item2["input_ids"]))

In [14]:
# # Check which parameters actually have gradients and which are frozen
# print("=== GRADIENT STATUS ===")
# frozen, trainable, zero_grad = [], [], []
# for name, param in model.named_parameters():
#     if not param.requires_grad:
#         frozen.append(name)
#     elif param.grad is None:
#         zero_grad.append(name)
#     else:
#         trainable.append(name)

# print(f"Frozen (requires_grad=False): {len(frozen)}")
# print(f"Trainable with grad:          {len(trainable)}")
# print(f"Trainable but grad is None:   {len(zero_grad)}")

# # Check classifier weights specifically
# print("\n=== CLASSIFIER WEIGHTS ===")
# print("classifier.weight:", model.classifier.weight.data)
# print("classifier.bias:  ", model.classifier.bias.data)

# # Check if encoder is actually processing input differently per sample
# # by hooking into the last encoder layer output
# hook_output = []
# def hook_fn(module, input, output):
#     hook_output.append(output[0].detach().cpu())

# hook = model.deberta.encoder.layer[-1].register_forward_hook(hook_fn)

# model.eval()
# with torch.no_grad():
#     for idx in crisis_indices[:2]:
#         item = val_ds[idx]
#         model(
#             input_ids=item["input_ids"].unsqueeze(0).to(model.device),
#             attention_mask=item["attention_mask"].unsqueeze(0).to(model.device),
#             token_type_ids=item["token_type_ids"].unsqueeze(0).to(model.device)
#         )

# hook.remove()

# print("\n=== LAST ENCODER LAYER OUTPUT ===")
# print("Sample 1 [CLS] vector (first 8 dims):", hook_output[0][0, 0, :8].tolist())
# print("Sample 2 [CLS] vector (first 8 dims):", hook_output[1][0, 0, :8].tolist())
# print("Are encoder outputs identical?", hook_output[0].equal(hook_output[1]))

In [15]:
# # Check actual model dtype and device
# print("Model dtype:", next(model.parameters()).dtype)
# print("Model device:", next(model.parameters()).device)

# # Check what bf16 actually does here
# print("bf16 flag in training_args:", training_args.bf16)
# print("fp16 flag in training_args:", training_args.fp16)

# # Force model to bfloat16 explicitly before training
# model = model.to(torch.bfloat16)
# print("Model dtype after cast:", next(model.parameters()).dtype)

# # Verify classifier is now bfloat16
# print("Classifier weight dtype:", model.classifier.weight.dtype)
# print("Classifier bias dtype:  ", model.classifier.bias.dtype)

In [16]:
# STEP 8: Training configuration
import glob
from transformers import TrainingArguments

# Detect existing checkpoints for resume
existing = sorted(glob.glob(f"{SAVE_DIR}/checkpoint-*"))
RESUME_FROM = None
print(f"Resume from: {RESUME_FROM}" if RESUME_FROM else "Starting fresh.")

training_args = TrainingArguments(
    output_dir=SAVE_DIR,

    # Schedule — lower LR than RoBERTa because full fine-tune (no LoRA)
    num_train_epochs=4,
    per_device_train_batch_size=32,     # DeBERTa-v3 is larger — 32 is safe on A100
    per_device_eval_batch_size=64,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,

    # Checkpointing
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="recall",     # recall is primary — never miss a crisis
    greater_is_better=True,
    save_total_limit=3,

    # Speed: Switched to bf16 because DeBERTa-v3 has issues with fp16 gradient scaling, and A100 supports bf16 natively.
    bf16=True,
    dataloader_num_workers=4,

    # Logging
    logging_steps=200,
    report_to="none",
)

print(f"Epochs       : {training_args.num_train_epochs}")
print(f"Train batch  : {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Best metric  : {training_args.metric_for_best_model}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting fresh.
Epochs       : 4
Train batch  : 32
Learning rate: 1e-05
Best metric  : recall


In [17]:
# STEP 9: Train
# Expected time: ~2 hours on A100
# Saves checkpoint to Drive after every epoch — safe to disconnect and resume
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print("Starting DeBERTa training...")
print("-" * 60)
trainer.train(resume_from_checkpoint=RESUME_FROM)
print("-" * 60)
print("Training complete.")


Starting DeBERTa training...
------------------------------------------------------------


Epoch,Training Loss,Validation Loss,Recall,Precision,F1
1,0.306844,0.325373,0.957342,0.813906,0.879816
2,0.291849,0.327647,0.963289,0.807717,0.878670
3,0.294966,0.337061,0.966046,0.798774,0.874483
4,0.294204,0.336354,0.965874,0.799486,0.874839


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

------------------------------------------------------------
Training complete.


In [18]:
# STEP 10: Save final model
import os
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Model saved to: {SAVE_DIR}")
print("\nContents:")
for f in sorted(os.listdir(SAVE_DIR)):
    fpath = os.path.join(SAVE_DIR, f)
    size  = os.path.getsize(fpath)/1e6 if os.path.isfile(fpath) else 0
    print(f"  {f:<45} {size:.1f} MB" if size > 0 else f"  {f}/")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/drive/MyDrive/Mapped_Data/safety_guardrail

Contents:
  checkpoint-11604/
  checkpoint-17406/
  checkpoint-23208/
  config.json                                   0.0 MB
  model.safetensors                             368.9 MB
  tokenizer.json                                8.3 MB
  tokenizer_config.json                         0.0 MB
  training_args.bin                             0.0 MB


In [19]:
# STEP 11: Evaluate on test set
import json
import numpy as np

print("Evaluating on test set (23,208 examples)...")
test_results = trainer.evaluate(test_ds)

recall    = test_results["eval_recall"]
precision = test_results["eval_precision"]
f1        = test_results["eval_f1"]

print("\n=== TEST SET RESULTS ===")
print(f"Recall    : {recall:.4f}   (target: > 0.80)")
print(f"Precision : {precision:.4f}  (target: > 0.65)")
print(f"F1        : {f1:.4f}")

print()
if recall >= 0.80 and precision >= 0.65:
    print("✅ PASS — both targets met.")
elif recall >= 0.80:
    print("✅ Recall target met.")
    print(f"⚠️  Precision {precision:.4f} below 0.65 — acceptable, recall is primary.")
else:
    print("⚠️  Recall below 0.80.")
    print("   Fallback: lower lr to 5e-6, increase epochs to 5, re-run from checkpoint.")

# Save for paper
results_path = f"{SAVE_DIR}/test_results.json"
with open(results_path, "w") as f:
    json.dump({k: float(v) for k, v in test_results.items()}, f, indent=2)
print(f"\nResults saved: {results_path}")


Evaluating on test set (23,208 examples)...



=== TEST SET RESULTS ===
Recall    : 0.9629   (target: > 0.80)
Precision : 0.7951  (target: > 0.65)
F1        : 0.8710

✅ PASS — both targets met.

Results saved: /content/drive/MyDrive/Mapped_Data/safety_guardrail/test_results.json


In [20]:
# STEP 12: Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

print("Computing confusion matrix...")
pred_output = trainer.predict(test_ds)
preds  = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(labels, preds, target_names=["crisis", "non-crisis"]))

cm = confusion_matrix(labels, preds)
print("=== CONFUSION MATRIX (rows=true, cols=pred) ===")
print(f"{'':>15}{'crisis':>12}{'non-crisis':>12}")
for i, row in enumerate(cm):
    name = ["crisis", "non-crisis"][i]
    print(f"{name:>15}" + "".join(f"{v:>12}" for v in row))

# False negatives = missed crises (top-right cell) — report explicitly
fn = cm[0][1]
total_crisis = cm[0].sum()
print(f"\nMissed crises (false negatives): {fn} / {total_crisis} ({100*fn/total_crisis:.1f}%)")

np.save(f"{SAVE_DIR}/confusion_matrix.npy", cm)
print(f"Confusion matrix saved to Drive.")


Computing confusion matrix...



=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

      crisis       0.80      0.96      0.87     11604
  non-crisis       0.95      0.75      0.84     11604

    accuracy                           0.86     23208
   macro avg       0.87      0.86      0.86     23208
weighted avg       0.87      0.86      0.86     23208

=== CONFUSION MATRIX (rows=true, cols=pred) ===
                     crisis  non-crisis
         crisis       11174         430
     non-crisis        2879        8725

Missed crises (false negatives): 430 / 11604 (3.7%)
Confusion matrix saved to Drive.


In [21]:
# STEP 13 Verify the saved model loads and produces non-collapsed predictions
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

ckpt = "/content/drive/MyDrive/Mapped_Data/safety_guardrail"
tok = AutoTokenizer.from_pretrained(ckpt)
m   = AutoModelForSequenceClassification.from_pretrained(ckpt, torch_dtype=torch.bfloat16).to("cuda").eval()

tests = [
    ("I want to kill myself and I have a plan.",          0),
    ("I think everyone would be better off without me.",  0),
    ("I'm stressed about my thesis deadline.",            1),
    ("this exam is literally killing me lol",             1),
]

print("SANITY CHECK — real guardrail inference:")
all_pass = True
for text, expected in tests:
    enc = tok(text, "This person is expressing suicidal ideation or intent to self-harm.",
              return_tensors="pt", truncation=True, max_length=256, padding="max_length")
    enc = {k: v.to("cuda") for k, v in enc.items()}
    with torch.no_grad():
        logits = m(**enc).logits
    pred = int(torch.argmax(logits, dim=-1).item())
    prob_crisis = torch.softmax(logits, dim=-1)[0, 0].item()
    status = "✅" if pred == expected else "❌"
    print(f"  {status} pred={'crisis' if pred==0 else 'non-crisis'} ({prob_crisis:.3f}) | {text[:60]}")
    if pred != expected:
        all_pass = False

print("\n✅ Checkpoint verified — safe to close Colab." if all_pass else "\n❌ At least one prediction wrong — do not close yet.")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

SANITY CHECK — real guardrail inference:
  ✅ pred=crisis (0.906) | I want to kill myself and I have a plan.
  ✅ pred=crisis (0.891) | I think everyone would be better off without me.
  ✅ pred=non-crisis (0.099) | I'm stressed about my thesis deadline.
  ✅ pred=non-crisis (0.209) | this exam is literally killing me lol

✅ Checkpoint verified — safe to close Colab.


## ✅ Training Complete — Next Steps

### 1. Download checkpoint from Drive
Navigate to `G:/My Drive/empathrag/safety_guardrail/` and copy the folder to:
```
models/safety_guardrail/
```
in your local repo.

### 2. Verify it loads locally
```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tok   = AutoTokenizer.from_pretrained("models/safety_guardrail")
model = AutoModelForSequenceClassification.from_pretrained("models/safety_guardrail").eval()

HYPOTHESIS = "This person is expressing suicidal ideation or intent to self-harm."
test_text  = "I think everyone would be better off without me."

enc  = tok(test_text, HYPOTHESIS, return_tensors="pt", truncation=True, max_length=256)
with torch.no_grad():
    prob_crisis = torch.softmax(model(**enc).logits, dim=-1)[0, 0].item()

print(f"Crisis probability: {prob_crisis:.3f}  (fires if > 0.5)")
```
Expected: crisis probability > 0.5 for that input.

### 3. Tell Ray the test_results.json numbers
Ray needs the recall and precision values to fill in the evaluation table in the paper.
